In [ ]:
import os
import json

In [ ]:
POSTFIX = '_07_27'
FILE_NAME = '_result_day_per_timept' + POSTFIX
OUT_NAME = 'compiled_<<TYPE>>_per_timestep' + POSTFIX
OUT_EXCEL = f'_result/{OUT_NAME}.xlsx'
files_data = []
for root, dirs, files in os.walk("./_result"):
    for file in files:
        if file.startswith(FILE_NAME) and file.endswith(".json"):
        # if file.startswith("_result_day_s40") and file.endswith(".json"):
            if file.startswith("_result_day_s40") : continue
            file_path = os.path.join(root, file)
            files_data.append(json.loads(open(file_path, "r").read()))

In [ ]:
import csv
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, Border, Side

ATTR_ORDER  = ['Temp', 'RH', 'WSpd', 'WDir']
MODEL_ORDER = ['Persistence24', 'Climatology', 'LSTM', 'LSTM-ED', 'LSTM-FC', 'LSTM-ED-FC', 'LSTM-ED-ATTN-FC', 'LSTM-ED-CNN-FC']
MODEL_ORDER_FC_COMP = ['LSTM', 'LSTM-FC', 'LSTM-ED', 'LSTM-ED-FC']

# col index -> True means higher-is-better (R2), False means lower-is-better (MAE/RMSE)
METRIC_COLS = {
    3: False, 4: False, 5: True,   # train MAE, RMSE, R2
    6: False, 7: False, 8: True,   # eval  MAE, RMSE, R2
    9: False, 10: False, 11: True, # test  MAE, RMSE, R2
}



In [ ]:
def outline_range(ws, min_col, max_col, min_row, max_row, style='medium'):
    """Draw a thick outline border around the rectangular range, preserving
    each cell's existing borders on non-boundary sides."""
    side = Side(style=style)
    for r in range(min_row, max_row + 1):
        for c in range(min_col, max_col + 1):
            cell = ws.cell(r, c)
            b = cell.border
            cell.border = Border(
                left=side if c == min_col else b.left,
                right=side if c == max_col else b.right,
                top=side if r == min_row else b.top,
                bottom=side if r == max_row else b.bottom,
            )

def outline_range_thin(ws, min_col, max_col, min_row, max_row, style='thin'):
    """Draw a thick outline border around the rectangular range, preserving
    each cell's existing borders on non-boundary sides."""
    side = Side(style=style)
    for r in range(min_row, max_row + 1):
        for c in range(min_col, max_col + 1):
            cell = ws.cell(r, c)
            cell.border = Border(side, side, side, side)

def _finalize_style(ws, num_format='0.000'):
    """Set every cell's font to Arial (preserving bold/size/italic/color) and format
    numeric data cells as numbers with 3 decimal places. Call right before wb.save()."""
    for row in ws.iter_rows():
        for cell in row:
            f = cell.font
            cell.font = Font(name='Arial', size=f.size, bold=f.bold, italic=f.italic, color=f.color)
            if isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                cell.number_format = num_format

def compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER, excel_path='_result/compiled_full_per_timestep.xlsx'):
    acc = {}

    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if model not in MODEL_ORDER: continue
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in ('train', 'eval', 'test'):
                    full = splits.get(split, {}).get('full')
                    if not full:
                        continue
                    key = (model, attr, split)
                    if key not in acc:
                        acc[key] = {'mae': [], 'rmse': [], 'r2': [], 'count': []}
                    acc[key]['mae'].append(full['mae'])
                    acc[key]['rmse'].append(full['rmse'])
                    acc[key]['r2'].append(full['r2'])
                    acc[key]['count'].append(full['count'])

    rows = []
    for (model, attr, split), vals in sorted(acc.items()):
        n = len(vals['mae'])
        rows.append({
            'model': model, 'attr': attr, 'split': split,
            'mae':    sum(vals['mae'])   / n,
            'rmse':   sum(vals['rmse'])  / n,
            'r2':     sum(vals['r2'])    / n,
            'count':  int(sum(vals['count']) / n),
            'n_files': n,
        })

    # with open(out_path, 'w', newline='') as f:
    #     writer = csv.DictWriter(f, fieldnames=['model', 'attr', 'split', 'mae', 'rmse', 'r2', 'count', 'n_files'])
    #     writer.writeheader()
    #     writer.writerows(rows)
    # print(f'Written {len(rows)} rows to {out_path}')

    # Build pivot keyed by (attr, model)
    pivot = {}
    for row in rows:
        key = (row['attr'], row['model'])
        if key not in pivot:
            pivot[key] = {'model': row['model'], 'attr': row['attr']}
        s = row['split']
        pivot[key][f'{s}-MAE']  = row['mae']
        pivot[key][f'{s}-RMSE'] = row['rmse']
        pivot[key][f'{s}-R2']   = row['r2']

    def sort_key(entry):
        attr, model = entry
        ai = ATTR_ORDER.index(attr)  if attr  in ATTR_ORDER  else len(ATTR_ORDER)
        mi = MODEL_ORDER.index(model) if model in MODEL_ORDER else len(MODEL_ORDER)
        return (ai, mi)

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    wb = Workbook()
    ws = wb.active

    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    bold   = Font(bold=True)

    # Row 1: merged group headers
    for col_letter, label in [('A', 'Model'), ('B', 'Attribute'),
                               ('C', 'Train'), ('F', 'Eval'), ('I', 'Test')]:
        ws[f'{col_letter}1'] = label
        ws[f'{col_letter}1'].alignment = center
        ws[f'{col_letter}1'].font = bold

    ws.merge_cells('A1:A2')
    ws.merge_cells('B1:B2')
    ws.merge_cells('C1:E1')
    ws.merge_cells('F1:H1')
    ws.merge_cells('I1:K1')

    # Row 2: sub-headers
    for col, header in enumerate(['MAE', 'RMSE', 'R2',
                                   'MAE', 'RMSE', 'R2',
                                   'MAE', 'RMSE', 'R2'], start=3):
        cell = ws.cell(2, col, header)
        cell.alignment = center
        cell.font = bold

    # Data rows starting at row 3
    DATA_START = 3
    for r_idx, row in enumerate(sorted_rows, start=DATA_START):
        ws.cell(r_idx, 1, row['model'])
        ws.cell(r_idx, 2, row['attr'])
        ws.cell(r_idx, 3, row.get('train-MAE'))
        ws.cell(r_idx, 4, row.get('train-RMSE'))
        ws.cell(r_idx, 5, row.get('train-R2'))
        ws.cell(r_idx, 6, row.get('eval-MAE'))
        ws.cell(r_idx, 7, row.get('eval-RMSE'))
        ws.cell(r_idx, 8, row.get('eval-R2'))
        ws.cell(r_idx, 9, row.get('test-MAE'))
        ws.cell(r_idx, 10, row.get('test-RMSE'))
        ws.cell(r_idx, 11, row.get('test-R2'))

    # Bold best value per metric per attr group
    attr_row_ranges = {}
    for i, row in enumerate(sorted_rows):
        attr = row['attr']
        excel_row = DATA_START + i
        if attr not in attr_row_ranges:
            attr_row_ranges[attr] = []
        attr_row_ranges[attr].append(excel_row)

    for attr, row_indices in attr_row_ranges.items():
        for col, higher_is_better in METRIC_COLS.items():
            candidates = [(ws.cell(r, col).value, r) for r in row_indices if ws.cell(r, col).value is not None]
            if not candidates:
                continue
            _, best_row = max(candidates) if higher_is_better else min(candidates)
            ws.cell(best_row, col).font = bold

    # Thick outline borders: every (column set x row set) block gets an outline.
    # Column sets: A | B | C-E (train) | F-H (eval) | I-K (test)
    # Row sets: 1-2 (header) then one block of 6 model rows per attribute.
    # outline_range_thin(ws, 1, 11, 1, 2 + len(ATTR_ORDER) * len(MODEL_ORDER))
    COL_SETS = [(1, 1), (2, 2), (3, 5), (6, 8), (9, 11)]
    ROW_SETS = [(1, 2)] + [(3 + i * len(MODEL_ORDER), 2 + (i + 1) * len(MODEL_ORDER)) for i in range(len(ATTR_ORDER))]
    for c0, c1 in COL_SETS:
        for r0, r1 in ROW_SETS:
            outline_range(ws, c0, c1, r0, r1)

    _finalize_style(ws)
    wb.save(excel_path)
    print(f'Written {len(sorted_rows)} rows to {excel_path}')

    return rows

rows = compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER, excel_path=OUT_EXCEL.replace('<<TYPE>>', 'full'))
rows = compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER_FC_COMP, excel_path=OUT_EXCEL.replace('<<TYPE>>', 'fc'))
# for r in rows:
#     print(r)

In [ ]:
# Per-day RMSE table: one combined RMSE per (model, attr, forecast-day), pooling
# train/eval/test (and all result files) by summed squared error (rmse**2 * count),
# then re-taking the root -> the RMSE over the union of all samples. (Train has the
# most samples so it dominates the pooled value; for an equal-weight blend, average
# the per-split RMSEs instead.)
# Requires: outline_range / outline_range_thin, ATTR_ORDER, MODEL_ORDER (cell above),
# and files_data (loader cell).

def compile_day_results(files_data, excel_path='_result/compiled_day_per_timestep.xlsx'):
    # (model, attr, day) -> pooled summed-squared-error + count across all splits/files
    acc = {}
    days_seen = set()
    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if model not in MODEL_ORDER: continue
            print(model)
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                # for split in ('train', 'eval', 'test'):
                for split in ['eval', 'test']:
                    full = splits.get(split, {}).get('full')
                    if not full:
                        continue
                    for day, dm in (full.get('days') or {}).items():
                        if not dm:
                            continue
                        rmse, count = dm.get('rmse'), dm.get('count')
                        if rmse is None or count is None:
                            continue
                        d = int(day)
                        days_seen.add(d)
                        a = acc.setdefault((model, attr, d), {'sum_se': 0.0, 'count': 0})
                        a['sum_se'] += (rmse ** 2) * count   # rmse^2 * n = summed squared error
                        a['count']  += count

    day_list = sorted(days_seen)

    # Pivot keyed by (attr, model): {day -> pooled RMSE}
    pivot = {}
    for (model, attr, d), a in acc.items():
        rmse = (a['sum_se'] / a['count']) ** 0.5 if a['count'] > 0 else None
        p = pivot.setdefault((attr, model), {'model': model, 'attr': attr, 'days': {}})
        p['days'][d] = rmse

    def sort_key(entry):
        attr, model = entry
        ai = ATTR_ORDER.index(attr)   if attr  in ATTR_ORDER  else len(ATTR_ORDER)
        mi = MODEL_ORDER.index(model) if model in MODEL_ORDER else len(MODEL_ORDER)
        return (ai, mi)

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    # ----- CSV -----
    # with open(out_path, 'w', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(['model', 'attr'] + [f'day{d}' for d in day_list])
    #     for row in sorted_rows:
    #         writer.writerow([row['model'], row['attr']] + [row['days'].get(d) for d in day_list])
    # print(f'Written {len(sorted_rows)} rows to {out_path}')

    # ----- Excel -----
    wb = Workbook()
    ws = wb.active
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    bold   = Font(bold=True)

    # Header row (single row): Model | Attribute | Day 0 | Day 1 | ...
    for col, label in [(1, 'Model'), (2, 'Attribute')]:
        c = ws.cell(1, col, label); c.alignment = center; c.font = bold
    for i, d in enumerate(day_list):
        c = ws.cell(1, 3 + i, f'Day {d}'); c.alignment = center; c.font = bold

    DATA_START = 2
    for r_idx, row in enumerate(sorted_rows, start=DATA_START):
        ws.cell(r_idx, 1, row['model'])
        ws.cell(r_idx, 2, row['attr'])
        for i, d in enumerate(day_list):
            ws.cell(r_idx, 3 + i, row['days'].get(d))

    # Bold the best (lowest RMSE) per day within each attribute group;
    # collect each attribute's contiguous row block for the borders.
    attr_rows = {}
    for i, row in enumerate(sorted_rows):
        attr_rows.setdefault(row['attr'], []).append(DATA_START + i)

    n_days = len(day_list)
    for r_indices in attr_rows.values():
        for j in range(n_days):
            col = 3 + j
            cands = [(ws.cell(r, col).value, r) for r in r_indices if ws.cell(r, col).value is not None]
            if not cands:
                continue
            _, best_row = min(cands)   # RMSE: lower is better
            ws.cell(best_row, col).font = bold

    # Borders: thin grid over the whole table, then medium outlines per
    # (column set x row set). Column sets: Model | Attribute | all Day cols.
    # Row sets: header, then one block per attribute.
    last_col  = 2 + n_days
    last_row  = DATA_START - 1 + len(sorted_rows)
    # outline_range_thin(ws, 1, last_col, 1, last_row)
    COL_SETS = [(1, 1), (2, 2), (3, last_col)]
    ROW_SETS = [(1, 1)] + [(min(rs), max(rs)) for rs in attr_rows.values()]
    for c0, c1 in COL_SETS:
        for r0, r1 in ROW_SETS:
            outline_range(ws, c0, c1, r0, r1)

    _finalize_style(ws)
    wb.save(excel_path)
    print(f'Written {len(sorted_rows)} rows to {excel_path}')
    return sorted_rows

day_rows = compile_day_results(files_data, excel_path=OUT_EXCEL.replace('<<TYPE>>', 'day'))

In [ ]:
# Per-station RMSE table: one combined RMSE per (attr, station, model), pooling all
# splits (train/eval/test) and result files for each station via summed squared error
# (rmse**2 * count) -> the RMSE over the union of that station's samples. Train
# stations pool their train+eval windows; held-out test stations use their test
# windows. Stations are rows (grouped by attribute), models are columns.
# Requires: outline_range / outline_range_thin, ATTR_ORDER, MODEL_ORDER, files_data.

def compile_station_results(files_data, excel_path='_result/compiled_station_per_timestep.xlsx'):
    # (model, attr, station) -> pooled summed-squared-error + count across all splits/files
    acc = {}
    models_seen = set()
    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in ('train', 'eval', 'test'):
                    sp = splits.get(split)
                    if not isinstance(sp, dict):
                        continue
                    for station, sm in (sp.get('stations') or {}).items():
                        if not sm:
                            continue
                        rmse, count = sm.get('rmse'), sm.get('count')
                        if rmse is None or count is None:
                            continue
                        a = acc.setdefault((model, attr, station), {'sum_se': 0.0, 'count': 0})
                        a['sum_se'] += (rmse ** 2) * count
                        a['count']  += count
                        models_seen.add(model)

    # Column order = models (MODEL_ORDER first, then any extras alphabetically)
    model_cols = [m for m in MODEL_ORDER if m in models_seen] + sorted(m for m in models_seen if m not in MODEL_ORDER)

    # Pivot keyed by (attr, station): {model -> pooled RMSE}
    pivot = {}
    for (model, attr, station), a in acc.items():
        rmse = (a['sum_se'] / a['count']) ** 0.5 if a['count'] > 0 else None
        p = pivot.setdefault((attr, station), {'attr': attr, 'station': station, 'models': {}})
        p['models'][model] = rmse

    def sort_key(entry):
        attr, station = entry
        ai = ATTR_ORDER.index(attr) if attr in ATTR_ORDER else len(ATTR_ORDER)
        return (ai, station)   # group by attribute, then station id

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    # ----- CSV -----
    # with open(out_path, 'w', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(['attr', 'station'] + model_cols)
    #     for row in sorted_rows:
    #         writer.writerow([row['attr'], row['station']] + [row['models'].get(m) for m in model_cols])
    # print(f'Written {len(sorted_rows)} rows to {out_path}')

    # ----- Excel -----
    wb = Workbook()
    ws = wb.active
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    bold   = Font(bold=True)

    # Header row: Attribute | Station | <model columns>
    for col, label in [(1, 'Attribute'), (2, 'Station')]:
        c = ws.cell(1, col, label); c.alignment = center; c.font = bold
    for i, m in enumerate(model_cols):
        c = ws.cell(1, 3 + i, m); c.alignment = center; c.font = bold

    DATA_START = 2
    for r_idx, row in enumerate(sorted_rows, start=DATA_START):
        ws.cell(r_idx, 1, row['attr'])
        ws.cell(r_idx, 2, row['station'])
        for i, m in enumerate(model_cols):
            ws.cell(r_idx, 3 + i, row['models'].get(m))

    # Bold the best (lowest RMSE) model in each station row
    n_models = len(model_cols)
    for r_idx in range(DATA_START, DATA_START + len(sorted_rows)):
        cands = [(ws.cell(r_idx, 3 + j).value, 3 + j) for j in range(n_models) if ws.cell(r_idx, 3 + j).value is not None]
        if not cands:
            continue
        _, best_col = min(cands)   # RMSE: lower is better
        ws.cell(r_idx, best_col).font = bold

    # Attribute row blocks (contiguous, since rows are grouped by attribute)
    attr_rows = {}
    for i, row in enumerate(sorted_rows):
        attr_rows.setdefault(row['attr'], []).append(DATA_START + i)

    # Borders: thin grid over the whole table, then medium outlines per
    # (column set x row set). Column sets: Attribute | Station | all model cols.
    last_col = 2 + n_models
    last_row = DATA_START - 1 + len(sorted_rows)
    outline_range_thin(ws, 1, last_col, 1, last_row)
    COL_SETS = [(1, 1), (2, 2), (3, last_col)]
    ROW_SETS = [(1, 1)] + [(min(rs), max(rs)) for rs in attr_rows.values()]
    for c0, c1 in COL_SETS:
        for r0, r1 in ROW_SETS:
            outline_range(ws, c0, c1, r0, r1)

    _finalize_style(ws)
    wb.save(excel_path)
    print(f'Written {len(sorted_rows)} rows to {excel_path}')
    return sorted_rows

station_rows = compile_station_results(files_data, excel_path=OUT_EXCEL.replace('<<TYPE>>', 'station'))